# Model and data observability

## 1. Setup

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F, Window, Row
session = get_active_session()

### 1A. Constants

In [ ]:
# Account info
DATABASE = "BD_AA_DEV"
STORAGE_SCHEMA = "SC_STORAGE_BMX_PS"
FEATURES_SCHEMA = "SC_FEATURES_BMX"
MODELS_SCHEMA = "SC_MODELS_BMX"

# Input data sources
BASELINE_DATA = f"{DATABASE}.{STORAGE_SCHEMA}.TRAIN_DATASET_CLEANED_VW"
INFERENCE_DATA = f"{DATABASE}.{STORAGE_SCHEMA}.INFERENCE_DATASET_CLEANED_VW"
BASELINE_PREDICTIONS = f"{DATABASE}.{STORAGE_SCHEMA}.TRAIN_PREDICTIONS_VW"
INFERENCE_PREDICTIONS = f"{DATABASE}.{STORAGE_SCHEMA}.INFERENCE_PREDICTIONS_VW"
INFERENCE_ACTUALS = f"{DATABASE}.{STORAGE_SCHEMA}.INFERENCE_ACTUALS_VW"

# Landing locations
DATA_DRIFT_HISTOGRAMS_BASELINE = f"{DATABASE}.{STORAGE_SCHEMA}.DA_DATA_DRIFT_HISTOGRAMS_BASELINE"
DATA_DRIFT_HISTOGRAMS_INFERENCE = f"{DATABASE}.{STORAGE_SCHEMA}.DA_DATA_DRIFT_HISTOGRAMS_INFERENCE"
DATA_DRIFT = f"{DATABASE}.{STORAGE_SCHEMA}.DA_DATA_DRIFT"
PRED_DRIFT_HISTOGRAMS_BASELINE = f"{DATABASE}.{STORAGE_SCHEMA}.DA_PREDICTION_DRIFT_HISTOGRAMS_BASELINE"
PRED_DRIFT_HISTOGRAMS_INFERENCE = f"{DATABASE}.{STORAGE_SCHEMA}.DA_PREDICTION_DRIFT_HISTOGRAMS_INFERENCE"
PRED_DRIFT = f"{DATABASE}.{STORAGE_SCHEMA}.DA_PREDICTION_DRIFT"
PERF_BASELINE = f"{DATABASE}.{STORAGE_SCHEMA}.DA_PERFORMANCE_BASELINE"
PERF_DRIFT = f"{DATABASE}.{STORAGE_SCHEMA}.DA_PERFORMANCE_DRIFT"

# Model specifics

MODEL_BASE = "UNI_BOX_REGRESSION_"
MODEL_NAME = MODEL_BASE + "PARTITIONED"
CUSTOMER_ID_COL = "CUSTOMER_ID"
BRAND_ID_COL = "BRAND_PRES_RET"
PRODUCT_ID_COL = "PROD_KEY"
AGG_COLS = ["STATS_NTILE_GROUP", "CUST_CATEGORY"]
TARGET_COL = "UNI_BOX_WEEK"
PREDICTION_COL = "PREDICTED_UNI_BOX_WEEK"
TIME_COL = "WEEK"

NON_FEATURE_COLS = {
    CUSTOMER_ID_COL, TIME_COL, BRAND_ID_COL,
    PRODUCT_ID_COL, TARGET_COL, *AGG_COLS,
}

NON_DRIFT_COLS = {'WEEK_OF_YEAR'}

# Feature drift settings
N_BINS = 20

# Performance metric settings
PERF_JOIN_KEYS = [CUSTOMER_ID_COL, TIME_COL, BRAND_ID_COL, PRODUCT_ID_COL]
PERF_METRIC_NAMES = ["wape", "rmse", "mae", "f1_binary"]


### 1B. Create landing tables


In [ ]:
# Create landing tables with explicit types

HISTOGRAM_SCHEMA = """
    RECORD_ID        VARCHAR(128) NOT NULL,
    MODEL_NAME       VARCHAR(64)  NOT NULL,
    MODEL_VERSION    VARCHAR(32)  NOT NULL,
    ENTITY_MAP       OBJECT,
    AGGREGATED_COL   VARCHAR(64)  NOT NULL,
    AGGREGATED_VALUE VARCHAR(64),
    METRIC_COL       VARCHAR(64)  NOT NULL,
    METRIC_MAP       OBJECT,
    CALMONTH         VARCHAR(6),
    LDTS             TIMESTAMP_LTZ(9) NOT NULL
"""

for tbl in [
    DATA_DRIFT_HISTOGRAMS_BASELINE, DATA_DRIFT_HISTOGRAMS_INFERENCE,
    PRED_DRIFT_HISTOGRAMS_BASELINE, PRED_DRIFT_HISTOGRAMS_INFERENCE,
]:
    session.sql(f"CREATE TABLE IF NOT EXISTS {tbl} ({HISTOGRAM_SCHEMA})").collect()


DRIFT_SCHEMA = """
    RECORD_ID          VARCHAR(128) NOT NULL,
    MODEL_NAME         VARCHAR(64)  NOT NULL,
    MODEL_VERSION      VARCHAR(32)  NOT NULL,
    ENTITY_MAP         OBJECT,
    AGGREGATED_COL     VARCHAR(64)  NOT NULL,
    AGGREGATED_VALUE   VARCHAR(64)  NOT NULL,
    METRIC_COL         VARCHAR(64)  NOT NULL,
    METRIC_VALUE       FLOAT,
    WARNING_THRESHOLD  FLOAT        NOT NULL,
    CRITICAL_THRESHOLD FLOAT        NOT NULL,
    ALERT_LEVEL        INT NOT NULL,
    BKCC               VARCHAR(5)   NOT NULL,
    CALMONTH           VARCHAR(6),
    LDTS               TIMESTAMP_LTZ(9) NOT NULL
"""

for tbl in [DATA_DRIFT, PRED_DRIFT]:
    session.sql(f"CREATE TABLE IF NOT EXISTS {tbl} ({DRIFT_SCHEMA})").collect()


PERF_SCHEMA = """
    RECORD_ID          VARCHAR(128) NOT NULL,
    MODEL_NAME         VARCHAR(64)  NOT NULL,
    MODEL_VERSION      VARCHAR(32)  NOT NULL,
    ENTITY_MAP         OBJECT,
    AGGREGATED_COL     VARCHAR(64)  NOT NULL,
    AGGREGATED_VALUE   VARCHAR(64)  NOT NULL,
    METRIC_COL         VARCHAR(64)  NOT NULL,
    METRIC_VALUE       FLOAT,
    METRIC_DRIFT       FLOAT,
    WARNING_THRESHOLD  FLOAT        NOT NULL,
    CRITICAL_THRESHOLD FLOAT        NOT NULL,
    ALERT_LEVEL        INT NOT NULL,
    BKCC               VARCHAR(5)   NOT NULL,
    CALMONTH           VARCHAR(6),
    LDTS               TIMESTAMP_LTZ(9) NOT NULL
"""

for tbl in [PERF_BASELINE, PERF_DRIFT]:
    session.sql(f"CREATE TABLE IF NOT EXISTS {tbl} ({PERF_SCHEMA})").collect()

print("Landing tables ready.")


### 1C. Functions

In [ ]:
def get_new_combos(baseline_table, inference_table, agg_col, metric_col, min_week=None):
    """Find new items in INFERENCE_PREDICTIONS not yet landed.

    Returns two separate things:
    1. new_baseline_versions — versions in INFERENCE_PREDICTIONS whose
       MODEL_VERSION does not yet appear in *baseline_table*.
    2. new_combos_list — (WEEK, MODEL_VERSION) pairs in
       INFERENCE_PREDICTIONS that are not yet present in *inference_table*.

    Parameters
    ----------
    baseline_table : str
        Fully-qualified baseline table name.
    inference_table : str
        Fully-qualified inference table name.
    agg_col : str
        Aggregation column being selected.
    metric_col : str
        Metric being selected.
    min_week : str or None
        If provided, only consider inference weeks >= this value.

    Returns
    -------
    tuple[list[str], list[Row]]
        (new_baseline_versions, new_combos_list)
    """
    all_combos = (
        session.table(INFERENCE_PREDICTIONS)
        .select(
            F.col(TIME_COL).alias("WEEK"),
            F.col("MODEL_VERSION"))
        .distinct()
    )
    if min_week is not None:
        all_combos = all_combos.filter(F.col(TIME_COL) >= min_week)

    all_versions = all_combos.select("MODEL_VERSION").distinct()

    # --- Check inference table for existing combos ---
    inf_landing_df = (
        session.table(inference_table)
        .filter(F.col("AGGREGATED_COL") == agg_col.lower())
        .filter(F.col("METRIC_COL") == metric_col)
    )

    existing_combos = (
        inf_landing_df
        .select(
            F.col("ENTITY_MAP")["week"].cast("STRING").alias("WEEK"),
            F.col("MODEL_VERSION"),
        )
        .distinct()
    )
    new_combos = all_combos.join(
        existing_combos, on=["WEEK", "MODEL_VERSION"], how="left_anti"
    )

    # --- Check baseline table for versions needing baseline ---
    base_landing_df = (
        session.table(baseline_table)
        .filter(F.col("AGGREGATED_COL") == agg_col.lower())
        .filter(F.col("METRIC_COL") == metric_col)
    )

    existing_baseline_versions = (
        base_landing_df
        .select("MODEL_VERSION")
        .distinct()
    )
    new_baseline_versions_df = all_versions.join(
        existing_baseline_versions, on="MODEL_VERSION", how="left_anti"
    )

    new_combos_list = new_combos.collect()
    new_baseline_versions = [row["MODEL_VERSION"] for row in new_baseline_versions_df.collect()]

    agg_label = f" [{agg_col}]" if agg_col else ""
    print(f"{inference_table}{agg_label}")
    print(f"  New (WEEK, MODEL_VERSION) combos: {len(new_combos_list)}")
    print(f"  Versions needing baseline: {new_baseline_versions}")

    return new_baseline_versions, new_combos_list


## 2. Data drift

### 2A. Segment stability

In [ ]:
def compute_segment_histograms(combos, data_source, target_table, agg_col):
    """Compute per-segment customer proportions and write to target_table.

    Parameters
    ----------
    combos : list[str] | list[Row]
        If list of str  -> model versions; reads all weeks from *data_source*.
        If list of Row  -> (WEEK, MODEL_VERSION) pairs; filters *data_source*
                           to one week per combo.
    data_source : str | dict[str, str]
        Single fully-qualified table name, or {version: table} mapping.
    target_table : str
        Fully-qualified table name to write results to.
    agg_col : str
        Column name to aggregate by (e.g. "STATS_NTILE_GROUP").
    """
    agg_col_lower = agg_col.lower()
    frames = []

    for combo in combos:
        # Baseline mode: each combo is a version string
        if isinstance(combo, str):
            version = combo
            src = data_source[version] if isinstance(data_source, dict) else data_source
            src_tbl = session.table(src)
        # Inference mode: each combo is a Row with WEEK and MODEL_VERSION
        elif isinstance(combo, Row):
            week, version = combo["WEEK"], combo["MODEL_VERSION"]
            src = data_source if isinstance(data_source, str) else data_source[version]
            src_tbl = session.table(src).filter(F.col(TIME_COL) == week)

        data_date_expr = F.dateadd(
            "week", F.col(TIME_COL) % F.lit(100) - F.lit(1),
            F.date_from_parts(F.floor(F.col(TIME_COL) / F.lit(100)), F.lit(1), F.lit(1)),
        )

        proportions = (
            src_tbl
            .group_by(TIME_COL, agg_col)
            .agg(F.count_distinct(CUSTOMER_ID_COL).alias("N_CUSTOMERS"))
            .with_column(
                "PROPORTION",
                F.col("N_CUSTOMERS")
                / F.sum("N_CUSTOMERS").over(Window.partition_by(TIME_COL)),
            )
            .with_column("DATA_DATE", data_date_expr)
            .select(
                F.sha2(
                    F.concat(
                        F.lit(MODEL_NAME), F.lit("||"),
                        F.lit(version), F.lit("||"),
                        F.object_construct(
                            F.lit("week"), F.col(TIME_COL),
                            F.lit("data_date"), F.col("DATA_DATE"),
                        ).cast("STRING"), F.lit("||"),
                        F.lit(agg_col_lower), F.lit("||"),
                        F.col(agg_col).cast("STRING"), F.lit("||"),
                        F.lit("psi"),
                    ),
                    256,
                ).alias("RECORD_ID"),
                F.lit(MODEL_NAME).alias("MODEL_NAME"),
                F.lit(version).alias("MODEL_VERSION"),
                F.object_construct(
                    F.lit("week"), F.col(TIME_COL),
                    F.lit("data_date"), F.col("DATA_DATE"),
                ).alias("ENTITY_MAP"),
                F.lit(agg_col_lower).alias("AGGREGATED_COL"),
                F.col(agg_col).cast("STRING").alias("AGGREGATED_VALUE"),
                F.lit("population_stability_index").alias("METRIC_COL"),
                F.object_construct(
                    F.lit("bin_count"), F.col("N_CUSTOMERS"),
                    F.lit("bin_prop"), F.col("PROPORTION"),
                ).alias("METRIC_MAP"),
                F.to_char(F.col("DATA_DATE"), F.lit("YYYYMM")).alias("CALMONTH"),
                F.current_timestamp().alias("LDTS"),
            )
        )
        frames.append(proportions)

    result = frames[0]
    for f in frames[1:]:
        result = result.union_all_by_name(f)

    result.write.mode("append").save_as_table(target_table)

    n_rows = session.table(target_table).count()
    print(f"Wrote {len(combos)} combo(s) for {agg_col}. {target_table} now has {n_rows} rows.")


In [ ]:
# Calculate segment proportions — loop over each aggregation column

new_combos_list = {}

for agg_col in AGG_COLS:
    new_baseline_versions, new_combos_list[agg_col] = get_new_combos(
        DATA_DRIFT_HISTOGRAMS_BASELINE, DATA_DRIFT_HISTOGRAMS_INFERENCE,
        agg_col=agg_col, metric_col="population_stability_index")

    if new_baseline_versions:
        baseline_source = {v: BASELINE_DATA for v in new_baseline_versions}
        compute_segment_histograms(
            new_baseline_versions, baseline_source,
            target_table=DATA_DRIFT_HISTOGRAMS_BASELINE, agg_col=agg_col)

    if new_combos_list[agg_col]:
        compute_segment_histograms(
            new_combos_list[agg_col], INFERENCE_DATA,
            target_table=DATA_DRIFT_HISTOGRAMS_INFERENCE, agg_col=agg_col)


In [ ]:
# Calculate PSI — loop over each aggregation column
EPSILON = F.lit(1e-10)

for agg_col in AGG_COLS:

    combo_keys = [F.lit(f'{row["WEEK"]}|{row["MODEL_VERSION"]}') for row in new_combos_list[agg_col]]

    # --- Baseline: average proportion per segment ---
    baseline = (
        session.table(DATA_DRIFT_HISTOGRAMS_BASELINE)
        .filter(F.col("METRIC_COL") == "population_stability_index")
        .filter(F.col("AGGREGATED_COL") == agg_col.lower())
        .select(
            F.col("AGGREGATED_VALUE"),
            F.col("METRIC_MAP")["bin_prop"].cast("DOUBLE").alias("PROPORTION"),
        )
        .group_by("AGGREGATED_VALUE")
        .agg(F.avg("PROPORTION").alias("BASELINE_PROPORTION"))
    )

    # --- Inference: only (WEEK, MODEL_VERSION) combos in new_combos_list ---
    inference = (
        session.table(DATA_DRIFT_HISTOGRAMS_INFERENCE)
        .filter(F.col("METRIC_COL") == "population_stability_index")
        .filter(F.col("AGGREGATED_COL") == agg_col.lower())
        .with_column("WEEK", F.col("ENTITY_MAP")["week"].cast("STRING"))
        .filter(
            F.concat(F.col("WEEK"), F.lit("|"), F.col("MODEL_VERSION"))
            .isin(combo_keys)
        )
        .select("WEEK", "MODEL_VERSION",
                F.col("AGGREGATED_VALUE"),
                F.col("METRIC_MAP")["bin_prop"].cast("DOUBLE").alias("INFERENCE_PROPORTION"))
    )

    # --- PSI per (WEEK, MODEL_VERSION) ---
    inf_prop = F.greatest(F.col("INFERENCE_PROPORTION"), EPSILON)
    base_prop = F.greatest(F.col("BASELINE_PROPORTION"), EPSILON)
    psi_component = (inf_prop - base_prop) * F.ln(inf_prop / base_prop)

    psi_df = (
        inference.join(baseline, on="AGGREGATED_VALUE")
        .with_column("PSI_COMPONENT", psi_component)
        .group_by("WEEK", "MODEL_VERSION")
        .agg(F.sum("PSI_COMPONENT").alias("PSI"))
        .with_column(
            "DATA_DATE",
            F.dateadd("week", F.col("WEEK") % F.lit(100) - F.lit(1),
                      F.date_from_parts(F.floor(F.col("WEEK") / F.lit(100)), F.lit(1), F.lit(1))),
        )
        .select(
            F.sha2(
                F.concat(
                    F.lit(MODEL_NAME), F.lit("||"),
                    F.col("MODEL_VERSION"), F.lit("||"),
                    F.object_construct(
                        F.lit("week"), F.col("WEEK"),
                        F.lit("data_date"), F.col("DATA_DATE"),
                    ).cast("STRING"), F.lit("||"),
                    F.lit(agg_col.lower()), F.lit("||"),
                    F.lit("full_model"),
                ),
                256,
            ).alias("RECORD_ID"),
            F.lit(MODEL_NAME).alias("MODEL_NAME"),
            F.col("MODEL_VERSION"),
            F.object_construct(
                F.lit("week"), F.col("WEEK"),
                F.lit("data_date"), F.col("DATA_DATE"),
            ).alias("ENTITY_MAP"),
            F.lit(agg_col.lower()).alias("AGGREGATED_COL"),
            F.lit("full_model").alias("AGGREGATED_VALUE"),
            F.lit("population_stability_index").alias("METRIC_COL"),
            F.col("PSI").cast("DOUBLE").alias("METRIC_VALUE"),
            F.lit(0.1).cast("DOUBLE").alias("WARNING_THRESHOLD"),
            F.lit(0.2).cast("DOUBLE").alias("CRITICAL_THRESHOLD"),
            F.when(F.col("PSI") > 0.2, F.lit(2))
             .when(F.col("PSI") > 0.1, F.lit(1))
             .otherwise(F.lit(0)).alias("ALERT_LEVEL"),
            F.lit("MXBEB").alias("BKCC"),
            F.to_char(F.col("DATA_DATE"), F.lit("YYYYMM")).alias("CALMONTH"),
            F.current_timestamp().alias("LDTS"),
        )
    )

    psi_df.write.mode("append").save_as_table(DATA_DRIFT)

    n_rows = session.table(DATA_DRIFT).count()
    print(f"PSI for {agg_col}: DATA_DRIFT now has {n_rows} rows.")


### 2B. Feature drift (Jensen-Shannon distance)


In [ ]:
def get_feature_bin_edges(src_tbl, model_version, is_baseline, agg_col):
    """Return per-feature, per-segment bin edges and the list of monitored features.

    In baseline mode, identifies numeric feature columns from the data
    source schema (excluding known non-feature columns), then computes
    bin edges separately for each segment.  Features with fewer than
    N_BINS distinct values get one bin per distinct value (edges at
    midpoints between sorted values, bookended by -Inf / +Inf).
    Features with N_BINS or more distinct values use N_BINS
    quantile-based breakpoints.

    In inference mode, retrieves the stored baseline edges from
    DATA_DRIFT_HISTOGRAMS_BASELINE filtered by model_version.

    Parameters
    ----------
    src_tbl : snowpark.DataFrame
        DataFrame pointing at the source data (possibly pre-filtered).
    model_version : str
        Model version string. In inference mode, used to filter baseline rows.
    is_baseline : bool
        True to compute edges from data; False to read from landing table.
    agg_col : str
        Column name to aggregate by (e.g. "STATS_NTILE_GROUP").

    Returns
    -------
    tuple[snowpark.DataFrame, list[str]]
        (bins_df, features) where bins_df has columns
        SEGMENT, FEATURE_NAME, BIN, BIN_LOW, BIN_HIGH;
        and features is the list of monitored feature names.
    """

    if is_baseline:

        # Identify numeric features from schema, excluding non-feature cols
        from snowflake.snowpark.types import (
            DecimalType, DoubleType, FloatType, IntegerType, LongType, ShortType,
        )
        numeric_types = (DecimalType, DoubleType, FloatType, IntegerType, LongType, ShortType)
        features = [
            field.name
            for field in src_tbl.schema.fields
            if isinstance(field.datatype, numeric_types)
            and field.name not in NON_FEATURE_COLS.union(NON_DRIFT_COLS)
        ]

        # Exclude dummy variables (features with <= 2 distinct observed values)
        distinct_counts = (
            src_tbl
            .select([F.count_distinct(F.col(f)).alias(f) for f in features])
            .collect()[0]
        )
        features = [f for f in features if distinct_counts[f] > 2]

        # Split into low-cardinality (< N_BINS distinct values globally)
        # and high-cardinality features so we only unpivot what each path needs.
        low_card_features = [f for f in features if distinct_counts[f] < N_BINS]
        high_card_features = [f for f in features if distinct_counts[f] >= N_BINS]

        edge_parts = []  # collect (SEGMENT, FEATURE_NAME, EDGE_IDX, EDGE_VALUE) DataFrames

        # ── Low-cardinality: midpoint edges between consecutive distinct values ──
        if low_card_features:
            lc_cast = [F.col(f).cast("FLOAT").alias(f) for f in low_card_features]
            lc_casted = src_tbl.select(F.col(agg_col), *lc_cast)

            lc_unpivoted = (
                lc_casted.unpivot(
                    "FEATURE_VALUE", "FEATURE_NAME",
                    [F.col(f) for f in low_card_features],
                )
                .select(
                    F.col(agg_col).cast("STRING").alias("AGGREGATED_VALUE"),
                    "FEATURE_NAME", "FEATURE_VALUE",
                )
                .filter(F.col("FEATURE_VALUE").is_not_null())
            )

            low_card_vals = lc_unpivoted.select(
                "AGGREGATED_VALUE", "FEATURE_NAME", "FEATURE_VALUE"
            ).distinct()

            val_rank = Window.partition_by("AGGREGATED_VALUE", "FEATURE_NAME").order_by("FEATURE_VALUE")
            low_card_ranked = low_card_vals.with_column(
                "VAL_IDX", F.row_number().over(val_rank)
            )

            curr = low_card_ranked.select(
                F.col("AGGREGATED_VALUE"), F.col("FEATURE_NAME"),
                F.col("VAL_IDX"),
                F.col("FEATURE_VALUE").alias("CURR_VAL"),
            )
            nxt = low_card_ranked.select(
                F.col("AGGREGATED_VALUE"), F.col("FEATURE_NAME"),
                (F.col("VAL_IDX") - F.lit(1)).alias("VAL_IDX"),
                F.col("FEATURE_VALUE").alias("NEXT_VAL"),
            )
            midpoints = (
                curr.join(nxt, on=["AGGREGATED_VALUE", "FEATURE_NAME", "VAL_IDX"])
                .select(
                    "AGGREGATED_VALUE", "FEATURE_NAME",
                    F.col("VAL_IDX").alias("EDGE_IDX"),
                    ((F.col("CURR_VAL") + F.col("NEXT_VAL")) / F.lit(2.0))
                        .cast("FLOAT").alias("EDGE_VALUE"),
                )
            )
            edge_parts.append(midpoints)

        # ── High-cardinality: quantile-based edges ──
        if high_card_features:
            hc_cast = [F.col(f).cast("FLOAT").alias(f) for f in high_card_features]
            hc_casted = src_tbl.select(F.col(agg_col), *hc_cast)

            hc_unpivoted = (
                hc_casted.unpivot(
                    "FEATURE_VALUE", "FEATURE_NAME",
                    [F.col(f) for f in high_card_features],
                )
                .select(
                    F.col(agg_col).cast("STRING").alias("AGGREGATED_VALUE"),
                    "FEATURE_NAME", "FEATURE_VALUE",
                )
                .filter(F.col("FEATURE_VALUE").is_not_null())
            )

            quantile_fracs = [i / N_BINS for i in range(1, N_BINS)]
            agg_exprs = [
                F.call_builtin(
                    "APPROX_PERCENTILE", F.col("FEATURE_VALUE"), F.lit(q)
                ).alias(f"P{idx}")
                for idx, q in enumerate(quantile_fracs)
            ]

            pct_wide = (
                hc_unpivoted
                .group_by("AGGREGATED_VALUE", "FEATURE_NAME")
                .agg(*agg_exprs)
                .cache_result()
            )

            p_cols = [F.col(f"P{idx}") for idx in range(len(quantile_fracs))]
            quantile_edges = (
                pct_wide
                .unpivot("EDGE_VALUE", "P_LABEL", p_cols)
                .select(
                    F.col("AGGREGATED_VALUE"),
                    F.col("FEATURE_NAME"),
                    (F.call_builtin("REGEXP_SUBSTR", F.col("P_LABEL"), F.lit("\\d+"))
                     .cast("INT") + F.lit(1)).alias("EDGE_IDX"),
                    F.col("EDGE_VALUE").cast("FLOAT").alias("EDGE_VALUE"),
                )
            )
            edge_parts.append(quantile_edges)

        # ── Combine interior edges from both paths ──
        interior_edges = edge_parts[0]
        for part in edge_parts[1:]:
            interior_edges = interior_edges.union_all_by_name(part)

        # Add sentinel edges: -Inf at index 0, +Inf at (max EDGE_IDX + 1)
        inf_val = float("inf")
        seg_feat = interior_edges.select("AGGREGATED_VALUE", "FEATURE_NAME").distinct()
        max_edge = (
            interior_edges
            .group_by("AGGREGATED_VALUE", "FEATURE_NAME")
            .agg(F.max("EDGE_IDX").alias("MAX_IDX"))
        )

        low_sentinel = seg_feat.select(
            F.col("AGGREGATED_VALUE"), F.col("FEATURE_NAME"),
            F.lit(0).alias("EDGE_IDX"),
            F.lit(-inf_val).cast("FLOAT").alias("EDGE_VALUE"),
        )
        high_sentinel = max_edge.select(
            F.col("AGGREGATED_VALUE"), F.col("FEATURE_NAME"),
            (F.col("MAX_IDX") + F.lit(1)).alias("EDGE_IDX"),
            F.lit(inf_val).cast("FLOAT").alias("EDGE_VALUE"),
        )

        all_edges = low_sentinel.union_all_by_name(
            interior_edges
        ).union_all_by_name(
            high_sentinel
        )

        # Self-join consecutive edges to form (BIN, BIN_LOW, BIN_HIGH)
        low = all_edges.select(
            F.col("AGGREGATED_VALUE"), F.col("FEATURE_NAME"),
            F.col("EDGE_IDX").alias("BIN"),
            F.col("EDGE_VALUE").alias("BIN_LOW"),
        )
        high = all_edges.select(
            F.col("AGGREGATED_VALUE"), F.col("FEATURE_NAME"),
            (F.col("EDGE_IDX") - F.lit(1)).alias("BIN"),
            F.col("EDGE_VALUE").alias("BIN_HIGH"),
        )

        bins = (
            low.join(high, on=["AGGREGATED_VALUE", "FEATURE_NAME", "BIN"])
            .filter(F.col("BIN") >= F.lit(1))
            .select("AGGREGATED_VALUE", "FEATURE_NAME", "BIN", "BIN_LOW", "BIN_HIGH")
        )

    else:
        # Inference mode: read edges from baseline table matching model_version
        bins = (
            session.table(DATA_DRIFT_HISTOGRAMS_BASELINE)
            .filter(
                (F.col("MODEL_VERSION") == model_version)
                & (F.col("MODEL_NAME") == MODEL_NAME)
                & (F.col("AGGREGATED_COL") == agg_col.lower())
                & (F.col("METRIC_COL") == "jensen-shannon")
            )
            .select(
                F.col("AGGREGATED_VALUE"),
                F.col("ENTITY_MAP")["feature_name"].cast("STRING").alias("FEATURE_NAME"),
                F.col("METRIC_MAP")["bin_number"].cast("INT").alias("BIN"),
                F.col("METRIC_MAP")["bin_low"].cast("FLOAT").alias("BIN_LOW"),
                F.col("METRIC_MAP")["bin_high"].cast("FLOAT").alias("BIN_HIGH"),
            )
            .distinct()
        )
        features = [
            row["FEATURE_NAME"]
            for row in bins.select("FEATURE_NAME").distinct().collect()
        ]

    return bins, features


In [ ]:
def compute_data_drift_histograms(combos, data_source, is_baseline, target_table, agg_col):
    """Compute per-feature, per-segment histograms and write to target_table.

    Uses bin edges from get_feature_bin_edges. In baseline mode, edges are
    computed from the data per segment; in inference mode, edges are
    retrieved from the stored baseline by model version.

    Every bin gets a row for every time period, even when no values fall in
    that bin (BIN_COUNT = 0), so downstream joins never produce null
    metadata columns.

    Parameters
    ----------
    combos : list[str] | list[Row]
        If list of str  -> model versions; reads all weeks from *data_source*.
        If list of Row  -> (WEEK, MODEL_VERSION) pairs; filters *data_source*
                           to one week per combo.
    data_source : str | dict[str, str]
        Single fully-qualified table name, or {version: table} mapping.
    is_baseline : bool
        True when computing baseline histograms (edges computed from data);
        False for inference (edges retrieved from baseline table).
    target_table : str
        Fully-qualified table name to write results to.
    agg_col : str
        Column name to aggregate by (e.g. "STATS_NTILE_GROUP").
    """
    agg_col_lower = agg_col.lower()
    frames = []

    for combo in combos:
        # Resolve data source and version
        if isinstance(combo, str):
            version = combo
            src = data_source[version] if isinstance(data_source, dict) else data_source
            src_tbl = session.table(src)
        elif isinstance(combo, Row):
            week, version = combo["WEEK"], combo["MODEL_VERSION"]
            src = data_source if isinstance(data_source, str) else data_source[version]
            src_tbl = session.table(src).filter(F.col(TIME_COL) == week)

        # Get bin edges and the list of features to monitor
        bins, features = get_feature_bin_edges(src_tbl, version, is_baseline, agg_col)

        # Unpivot source data into (TIME, SEGMENT, FEATURE_NAME, FEATURE_VALUE)
        cast_exprs = [F.col(f).cast("FLOAT").alias(f) for f in features]
        src_casted = src_tbl.select(
            F.col(TIME_COL), F.col(agg_col), *cast_exprs
        )
        unpivoted = src_casted.unpivot(
            "FEATURE_VALUE", "FEATURE_NAME",
            [F.col(feat) for feat in features],
        ).select(
            F.col(TIME_COL),
            F.col(agg_col).cast("STRING").alias("AGGREGATED_VALUE"),
            "FEATURE_NAME", "FEATURE_VALUE",
        ).filter(F.col("FEATURE_VALUE").is_not_null())

        # Count values per bin (only bins that have data)
        actual_counts = (
            unpivoted
            .join(bins, on=["FEATURE_NAME", "AGGREGATED_VALUE"])
            .filter(
                (F.col("FEATURE_VALUE") >= F.col("BIN_LOW"))
                & (F.col("FEATURE_VALUE") < F.col("BIN_HIGH"))
            )
            .group_by(TIME_COL, "AGGREGATED_VALUE", "FEATURE_NAME", "BIN", "BIN_LOW", "BIN_HIGH")
            .agg(F.count("*").alias("BIN_COUNT"))
        )

        # Build scaffold: every (TIME x bin) combination so empty bins get a row
        distinct_times = src_casted.select(F.col(TIME_COL)).distinct()
        scaffold = distinct_times.cross_join(bins)

        binned = (
            scaffold.join(
                actual_counts,
                on=[TIME_COL, "AGGREGATED_VALUE", "FEATURE_NAME", "BIN", "BIN_LOW", "BIN_HIGH"],
                how="left",
            )
            .with_column("BIN_COUNT", F.coalesce(F.col("BIN_COUNT"), F.lit(0)))
        )

        # Compute DATA_DATE from WEEK
        data_date_expr = F.dateadd(
            "week",
            F.col(TIME_COL) % F.lit(100) - F.lit(1),
            F.date_from_parts(
                F.floor(F.col(TIME_COL) / F.lit(100)),
                F.lit(1), F.lit(1),
            ),
        )

        # Build long-format output
        histograms = (
            binned
            .with_column("DATA_DATE", data_date_expr)
            .select(
                F.sha2(
                    F.concat(
                        F.lit(MODEL_NAME), F.lit("||"),
                        F.lit(version), F.lit("||"),
                        F.object_construct(
                            F.lit("feature_name"), F.col("FEATURE_NAME"),
                            F.lit("week"), F.col(TIME_COL),
                            F.lit("data_date"), F.col("DATA_DATE"),
                        ).cast("STRING"), F.lit("||"),
                        F.lit(agg_col_lower), F.lit("||"),
                        F.col("AGGREGATED_VALUE").cast("STRING"), F.lit("||"),
                        F.lit("jsd"), F.lit("||"),
                        F.col("BIN").cast("STRING"),
                    ),
                    256,
                ).alias("RECORD_ID"),
                F.lit(MODEL_NAME).alias("MODEL_NAME"),
                F.lit(version).alias("MODEL_VERSION"),
                F.object_construct(
                    F.lit("feature_name"), F.col("FEATURE_NAME"),
                    F.lit("week"), F.col(TIME_COL),
                    F.lit("data_date"), F.col("DATA_DATE"),
                ).alias("ENTITY_MAP"),
                F.lit(agg_col_lower).alias("AGGREGATED_COL"),
                F.col("AGGREGATED_VALUE"),
                F.lit("jensen-shannon").alias("METRIC_COL"),
                F.object_construct(
                    F.lit("bin_number"), F.col("BIN"),
                    F.lit("bin_low"), F.col("BIN_LOW"),
                    F.lit("bin_high"), F.col("BIN_HIGH"),
                    F.lit("bin_count"), F.col("BIN_COUNT"),
                ).alias("METRIC_MAP"),
                F.to_char(F.col("DATA_DATE"), F.lit("YYYYMM")).alias("CALMONTH"),
                F.current_timestamp().alias("LDTS"),
            )
        )
        frames.append(histograms)

    result = frames[0]
    for f in frames[1:]:
        result = result.union_all_by_name(f)

    result.write.mode("append").save_as_table(target_table)

    n_rows = session.table(target_table).count()
    print(f"Wrote {len(combos)} combo(s) for {agg_col}. {target_table} now has {n_rows} rows.")


In [ ]:
# Calculate feature histograms — loop over each aggregation column
new_combos_list = {}
for agg_col in AGG_COLS:
    new_baseline_versions, new_combos_list[agg_col] = get_new_combos(
        DATA_DRIFT_HISTOGRAMS_BASELINE, DATA_DRIFT_HISTOGRAMS_INFERENCE,
        agg_col=agg_col, metric_col="jensen-shannon")

    if new_baseline_versions:
        baseline_source = {v: BASELINE_DATA for v in new_baseline_versions}
        compute_data_drift_histograms(
            new_baseline_versions, baseline_source,
            is_baseline=True, target_table=DATA_DRIFT_HISTOGRAMS_BASELINE, agg_col=agg_col)

    if new_combos_list[agg_col]:
        compute_data_drift_histograms(
            new_combos_list[agg_col], INFERENCE_DATA,
            is_baseline=False, target_table=DATA_DRIFT_HISTOGRAMS_INFERENCE, agg_col=agg_col)


In [ ]:
# Calculate Jensen-Shannon Distance from feature histograms — loop over each aggregation column
EPSILON = F.lit(1e-10)

for agg_col in AGG_COLS:
    agg_col_lower = agg_col.lower()

    combo_keys = [F.lit(f'{row["WEEK"]}|{row["MODEL_VERSION"]}') for row in new_combos_list[agg_col]]

    # --- JSD rows for this agg_col ---
    jsd_common_filters = [
        F.col("METRIC_COL") == "jensen-shannon",
        F.col("AGGREGATED_COL") == agg_col_lower,
    ]

    def _add_jsd_cols(df):
        return (df
            .with_column("WEEK", F.col("ENTITY_MAP")["week"].cast("STRING"))
            .with_column("FEATURE_NAME", F.col("ENTITY_MAP")["feature_name"].cast("STRING"))
            .with_column("BIN", F.col("METRIC_MAP")["bin_number"].cast("INT"))
            .with_column("BIN_COUNT", F.col("METRIC_MAP")["bin_count"].cast("INT"))
        )

    baseline_hist = _add_jsd_cols(
        session.table(DATA_DRIFT_HISTOGRAMS_BASELINE)
        .filter(jsd_common_filters[0]).filter(jsd_common_filters[1])
    )
    inference_hist = _add_jsd_cols(
        session.table(DATA_DRIFT_HISTOGRAMS_INFERENCE)
        .filter(jsd_common_filters[0]).filter(jsd_common_filters[1])
    )

    # --- Baseline: average bin probability per (SEGMENT, FEATURE_NAME, BIN) ---
    baseline_total = F.sum("BIN_COUNT").over(
        Window.partition_by("WEEK", "AGGREGATED_VALUE", "FEATURE_NAME")
    )
    baseline = (
        baseline_hist
        .with_column("BIN_PROB", F.col("BIN_COUNT") / F.greatest(baseline_total, F.lit(1.0)))
        .group_by("AGGREGATED_VALUE", "FEATURE_NAME", "BIN")
        .agg(F.avg("BIN_PROB").alias("P"))
    )

    # --- Inference: probabilities for new combos only ---
    inf_total = F.sum("BIN_COUNT").over(
        Window.partition_by("WEEK", "MODEL_VERSION", "AGGREGATED_VALUE", "FEATURE_NAME")
    )
    inference = (
        inference_hist
        .filter(
            F.concat(F.col("WEEK"), F.lit("|"), F.col("MODEL_VERSION"))
            .isin(combo_keys)
        )
        .with_column("Q", F.col("BIN_COUNT") / inf_total)
        .select("WEEK", "MODEL_VERSION", "AGGREGATED_VALUE", "FEATURE_NAME", "BIN",
                F.col("Q"))
    )

    # --- Full outer join on BIN so bins in only one distribution contribute ---
    joined = inference.join(
        baseline,
        on=["AGGREGATED_VALUE", "FEATURE_NAME", "BIN"],
        how="full",
    )

    # Apply epsilon smoothing to avoid log(0)
    p = F.greatest(F.coalesce(F.col("P"), F.lit(0.0)), EPSILON)
    q = F.greatest(F.coalesce(F.col("Q"), F.lit(0.0)), EPSILON)
    m = (p + q) / F.lit(2.0)

    # KL(P||M) and KL(Q||M) components
    kl_p_m = p * F.ln(p / m)
    kl_q_m = q * F.ln(q / m)

    # JSD = sqrt(0.5 * KL(P||M) + 0.5 * KL(Q||M))
    jsd_df = (
        joined
        .with_column("KL_P_M", kl_p_m)
        .with_column("KL_Q_M", kl_q_m)
        .group_by("WEEK", "MODEL_VERSION", "AGGREGATED_VALUE", "FEATURE_NAME")
        .agg(
            F.sum("KL_P_M").alias("SUM_KL_P_M"),
            F.sum("KL_Q_M").alias("SUM_KL_Q_M"),
        )
        .with_column(
            "JS_DISTANCE",
            F.sqrt(F.lit(0.5) * F.col("SUM_KL_P_M") + F.lit(0.5) * F.col("SUM_KL_Q_M")),
        )
        .with_column(
            "DATA_DATE",
            F.dateadd("week", F.col("WEEK") % F.lit(100) - F.lit(1),
                      F.date_from_parts(F.floor(F.col("WEEK") / F.lit(100)), F.lit(1), F.lit(1))),
        )
        .select(
            F.sha2(
                F.concat(
                    F.lit(MODEL_NAME), F.lit("||"),
                    F.col("MODEL_VERSION"), F.lit("||"),
                    F.object_construct(
                        F.lit("feature_name"), F.col("FEATURE_NAME"),
                        F.lit("week"), F.col("WEEK"),
                        F.lit("data_date"), F.col("DATA_DATE"),
                    ).cast("STRING"), F.lit("||"),
                    F.lit(agg_col_lower), F.lit("||"),
                    F.col("AGGREGATED_VALUE").cast("STRING"),
                ),
                256,
            ).alias("RECORD_ID"),
            F.lit(MODEL_NAME).alias("MODEL_NAME"),
            F.col("MODEL_VERSION"),
            F.object_construct(
                F.lit("feature_name"), F.col("FEATURE_NAME"),
                F.lit("week"), F.col("WEEK"),
                F.lit("data_date"), F.col("DATA_DATE"),
            ).alias("ENTITY_MAP"),
            F.lit(agg_col_lower).alias("AGGREGATED_COL"),
            F.col("AGGREGATED_VALUE").cast("STRING").alias("AGGREGATED_VALUE"),
            F.lit("jensen-shannon").alias("METRIC_COL"),
            F.col("JS_DISTANCE").cast("DOUBLE").alias("METRIC_VALUE"),
            F.lit(0.2).cast("DOUBLE").alias("WARNING_THRESHOLD"),
            F.lit(0.45).cast("DOUBLE").alias("CRITICAL_THRESHOLD"),
            F.when(F.col("JS_DISTANCE") > 0.45, F.lit(2))
             .when(F.col("JS_DISTANCE") > 0.2, F.lit(1))
             .otherwise(F.lit(0)).alias("ALERT_LEVEL"),
            F.lit("MXBEB").alias("BKCC"),
            F.to_char(F.col("DATA_DATE"), F.lit("YYYYMM")).alias("CALMONTH"),
            F.current_timestamp().alias("LDTS"),
        )
    )

    jsd_df.write.mode("append").save_as_table(DATA_DRIFT)

    n_rows = session.table(DATA_DRIFT).count()
    print(f"JSD for {agg_col}: DATA_DRIFT now has {n_rows} rows.")


## 3. Prediction drift

In [ ]:
def get_prediction_bin_edges(src_tbl, model_version, is_baseline, agg_col):
    """Return per-segment bin edges for the prediction column.

    In baseline mode, computes N_BINS quantile-based bin edges from
    PREDICTION_COL, separately for each segment defined by *agg_col*.

    In inference mode, retrieves the stored baseline edges from
    PRED_DRIFT_HISTOGRAMS_BASELINE filtered by model_version.

    Parameters
    ----------
    src_tbl : snowpark.DataFrame
        DataFrame pointing at the source data (possibly pre-filtered).
    model_version : str
        Model version string.
    is_baseline : bool
        True to compute edges from data; False to read from landing table.
    agg_col : str
        Column name to aggregate by (e.g. "STATS_NTILE_GROUP").

    Returns
    -------
    snowpark.DataFrame
        bins_df with columns AGGREGATED_VALUE, BIN, BIN_LOW, BIN_HIGH.
    """

    if is_baseline:
        quantile_fracs = [i / N_BINS for i in range(1, N_BINS)]
        agg_exprs = [
            F.call_builtin(
                "APPROX_PERCENTILE", F.col(PREDICTION_COL).cast("FLOAT"), F.lit(q)
            ).alias(f"P{idx}")
            for idx, q in enumerate(quantile_fracs)
        ]

        pct_wide = (
            src_tbl
            .group_by(F.col(agg_col).cast("STRING").alias("AGGREGATED_VALUE"))
            .agg(*agg_exprs)
            .cache_result()
        )

        p_cols = [F.col(f"P{idx}") for idx in range(len(quantile_fracs))]
        interior_edges = (
            pct_wide
            .unpivot("EDGE_VALUE", "P_LABEL", p_cols)
            .select(
                F.col("AGGREGATED_VALUE"),
                (F.call_builtin("REGEXP_SUBSTR", F.col("P_LABEL"), F.lit("\\d+"))
                 .cast("INT") + F.lit(1)).alias("EDGE_IDX"),
                F.col("EDGE_VALUE").cast("FLOAT").alias("EDGE_VALUE"),
            )
        )

        # Add sentinel edges: -Inf at index 0, +Inf at (max EDGE_IDX + 1)
        inf_val = float("inf")
        segments = interior_edges.select("AGGREGATED_VALUE").distinct()
        max_edge = (
            interior_edges
            .group_by("AGGREGATED_VALUE")
            .agg(F.max("EDGE_IDX").alias("MAX_IDX"))
        )

        low_sentinel = segments.select(
            F.col("AGGREGATED_VALUE"),
            F.lit(0).alias("EDGE_IDX"),
            F.lit(-inf_val).cast("FLOAT").alias("EDGE_VALUE"),
        )
        high_sentinel = max_edge.select(
            F.col("AGGREGATED_VALUE"),
            (F.col("MAX_IDX") + F.lit(1)).alias("EDGE_IDX"),
            F.lit(inf_val).cast("FLOAT").alias("EDGE_VALUE"),
        )

        all_edges = low_sentinel.union_all_by_name(
            interior_edges
        ).union_all_by_name(
            high_sentinel
        )

        # Self-join consecutive edges to form (BIN, BIN_LOW, BIN_HIGH)
        low = all_edges.select(
            F.col("AGGREGATED_VALUE"),
            F.col("EDGE_IDX").alias("BIN"),
            F.col("EDGE_VALUE").alias("BIN_LOW"),
        )
        high = all_edges.select(
            F.col("AGGREGATED_VALUE"),
            (F.col("EDGE_IDX") - F.lit(1)).alias("BIN"),
            F.col("EDGE_VALUE").alias("BIN_HIGH"),
        )

        bins = (
            low.join(high, on=["AGGREGATED_VALUE", "BIN"])
            .filter(F.col("BIN") >= F.lit(1))
            .select("AGGREGATED_VALUE", "BIN", "BIN_LOW", "BIN_HIGH")
        )

    else:
        # Inference mode: read edges from baseline table matching model_version
        bins = (
            session.table(PRED_DRIFT_HISTOGRAMS_BASELINE)
            .filter(
                (F.col("MODEL_VERSION") == model_version)
                & (F.col("MODEL_NAME") == MODEL_NAME)
                & (F.col("AGGREGATED_COL") == agg_col.lower())
                & (F.col("METRIC_COL") == "jensen-shannon")
            )
            .select(
                F.col("AGGREGATED_VALUE"),
                F.col("METRIC_MAP")["bin_number"].cast("INT").alias("BIN"),
                F.col("METRIC_MAP")["bin_low"].cast("FLOAT").alias("BIN_LOW"),
                F.col("METRIC_MAP")["bin_high"].cast("FLOAT").alias("BIN_HIGH"),
            )
            .distinct()
        )

    return bins


In [ ]:
def compute_prediction_drift_histograms(combos, data_source, is_baseline, target_table, agg_col):
    """Compute prediction histograms per segment and write to target_table.

    Parameters
    ----------
    combos : list[str] | list[Row]
        If list of str  -> model versions; reads all weeks from *data_source*.
        If list of Row  -> (WEEK, MODEL_VERSION) pairs.
    data_source : str | dict[str, str]
        Single fully-qualified table name, or {version: table} mapping.
    is_baseline : bool
        True when computing baseline histograms (edges computed from data);
        False for inference (edges retrieved from baseline table).
    target_table : str
        Fully-qualified table name to write results to.
    agg_col : str
        Column name to aggregate by (e.g. "STATS_NTILE_GROUP").
    """
    agg_col_lower = agg_col.lower()
    frames = []

    for combo in combos:
        # Resolve data source and version
        if isinstance(combo, str):
            version = combo
            src = data_source[version] if isinstance(data_source, dict) else data_source
            src_tbl = session.table(src)
        elif isinstance(combo, Row):
            week, version = combo["WEEK"], combo["MODEL_VERSION"]
            src = data_source if isinstance(data_source, str) else data_source[version]
            src_tbl = session.table(src).filter(F.col(TIME_COL) == week)

        # Get bin edges
        bins = get_prediction_bin_edges(src_tbl, version, is_baseline, agg_col)

        # Bin the prediction column directly (no unpivot needed)
        pred_values = (
            src_tbl
            .select(
                F.col(TIME_COL),
                F.col(agg_col).cast("STRING").alias("AGGREGATED_VALUE"),
                F.col(PREDICTION_COL).cast("FLOAT").alias("PRED_VALUE"),
            )
            .filter(F.col("PRED_VALUE").is_not_null())
        )

        actual_counts = (
            pred_values
            .join(bins, on=["AGGREGATED_VALUE"])
            .filter(
                (F.col("PRED_VALUE") >= F.col("BIN_LOW"))
                & (F.col("PRED_VALUE") < F.col("BIN_HIGH"))
            )
            .group_by(TIME_COL, "AGGREGATED_VALUE", "BIN", "BIN_LOW", "BIN_HIGH")
            .agg(F.count("*").alias("BIN_COUNT"))
        )

        # Scaffold: every (TIME x bin) so empty bins get a row
        distinct_times = src_tbl.select(F.col(TIME_COL)).distinct()
        scaffold = distinct_times.cross_join(bins)

        binned = (
            scaffold.join(
                actual_counts,
                on=[TIME_COL, "AGGREGATED_VALUE", "BIN", "BIN_LOW", "BIN_HIGH"],
                how="left",
            )
            .with_column("BIN_COUNT", F.coalesce(F.col("BIN_COUNT"), F.lit(0)))
        )

        data_date_expr = F.dateadd(
            "week",
            F.col(TIME_COL) % F.lit(100) - F.lit(1),
            F.date_from_parts(
                F.floor(F.col(TIME_COL) / F.lit(100)),
                F.lit(1), F.lit(1),
            ),
        )

        histograms = (
            binned
            .with_column("DATA_DATE", data_date_expr)
            .select(
                F.sha2(
                    F.concat(
                        F.lit(MODEL_NAME), F.lit("||"),
                        F.lit(version), F.lit("||"),
                        F.object_construct(
                            F.lit("prediction_name"), F.lit(PREDICTION_COL),
                            F.lit("week"), F.col(TIME_COL),
                            F.lit("data_date"), F.col("DATA_DATE"),
                        ).cast("STRING"), F.lit("||"),
                        F.lit(agg_col_lower), F.lit("||"),
                        F.col("AGGREGATED_VALUE").cast("STRING"), F.lit("||"),
                        F.lit("jsd"), F.lit("||"),
                        F.col("BIN").cast("STRING"),
                    ),
                    256,
                ).alias("RECORD_ID"),
                F.lit(MODEL_NAME).alias("MODEL_NAME"),
                F.lit(version).alias("MODEL_VERSION"),
                F.object_construct(
                    F.lit("prediction_name"), F.lit(PREDICTION_COL),
                    F.lit("week"), F.col(TIME_COL),
                    F.lit("data_date"), F.col("DATA_DATE"),
                ).alias("ENTITY_MAP"),
                F.lit(agg_col_lower).alias("AGGREGATED_COL"),
                F.col("AGGREGATED_VALUE"),
                F.lit("jensen-shannon").alias("METRIC_COL"),
                F.object_construct(
                    F.lit("bin_number"), F.col("BIN"),
                    F.lit("bin_low"), F.col("BIN_LOW"),
                    F.lit("bin_high"), F.col("BIN_HIGH"),
                    F.lit("bin_count"), F.col("BIN_COUNT"),
                ).alias("METRIC_MAP"),
                F.to_char(F.col("DATA_DATE"), F.lit("YYYYMM")).alias("CALMONTH"),
                F.current_timestamp().alias("LDTS"),
            )
        )
        frames.append(histograms)

    result = frames[0]
    for f in frames[1:]:
        result = result.union_all_by_name(f)

    result.write.mode("append").save_as_table(target_table)

    n_rows = session.table(target_table).count()
    print(f"Wrote {len(combos)} combo(s) for {agg_col}. {target_table} now has {n_rows} rows.")


In [ ]:
# Calculate prediction histograms — loop over each aggregation column
pred_new_combos_list = {}
for agg_col in AGG_COLS:
    new_baseline_versions, pred_new_combos_list[agg_col] = get_new_combos(
        PRED_DRIFT_HISTOGRAMS_BASELINE, PRED_DRIFT_HISTOGRAMS_INFERENCE,
        agg_col=agg_col, metric_col="jensen-shannon")

    if new_baseline_versions:
        baseline_source = {v: BASELINE_PREDICTIONS for v in new_baseline_versions}
        compute_prediction_drift_histograms(
            new_baseline_versions, baseline_source,
            is_baseline=True, target_table=PRED_DRIFT_HISTOGRAMS_BASELINE, agg_col=agg_col)

    if pred_new_combos_list[agg_col]:
        compute_prediction_drift_histograms(
            pred_new_combos_list[agg_col], INFERENCE_PREDICTIONS,
            is_baseline=False, target_table=PRED_DRIFT_HISTOGRAMS_INFERENCE, agg_col=agg_col)


In [ ]:
# Calculate Jensen-Shannon Distance from prediction histograms — loop over each aggregation column
EPSILON = F.lit(1e-10)

for agg_col in AGG_COLS:
    agg_col_lower = agg_col.lower()

    combo_keys = [F.lit(f'{row["WEEK"]}|{row["MODEL_VERSION"]}') for row in pred_new_combos_list[agg_col]]

    # --- JSD rows for this agg_col ---
    jsd_common_filters = [
        F.col("METRIC_COL") == "jensen-shannon",
        F.col("AGGREGATED_COL") == agg_col_lower,
    ]

    def _add_jsd_cols(df):
        return (df
            .with_column("WEEK", F.col("ENTITY_MAP")["week"].cast("STRING"))
            .with_column("BIN", F.col("METRIC_MAP")["bin_number"].cast("INT"))
            .with_column("BIN_COUNT", F.col("METRIC_MAP")["bin_count"].cast("INT"))
        )

    baseline_hist = _add_jsd_cols(
        session.table(PRED_DRIFT_HISTOGRAMS_BASELINE)
        .filter(jsd_common_filters[0]).filter(jsd_common_filters[1])
    )
    inference_hist = _add_jsd_cols(
        session.table(PRED_DRIFT_HISTOGRAMS_INFERENCE)
        .filter(jsd_common_filters[0]).filter(jsd_common_filters[1])
    )

    # --- Baseline: average bin probability per (SEGMENT, BIN) ---
    baseline_total = F.sum("BIN_COUNT").over(
        Window.partition_by("WEEK", "AGGREGATED_VALUE")
    )
    baseline = (
        baseline_hist
        .with_column("BIN_PROB", F.col("BIN_COUNT") / F.greatest(baseline_total, F.lit(1.0)))
        .group_by("AGGREGATED_VALUE", "BIN")
        .agg(F.avg("BIN_PROB").alias("P"))
    )

    # --- Inference: probabilities for new combos only ---
    inf_total = F.sum("BIN_COUNT").over(
        Window.partition_by("WEEK", "MODEL_VERSION", "AGGREGATED_VALUE")
    )
    inference = (
        inference_hist
        .filter(
            F.concat(F.col("WEEK"), F.lit("|"), F.col("MODEL_VERSION"))
            .isin(combo_keys)
        )
        .with_column("Q", F.col("BIN_COUNT") / inf_total)
        .select("WEEK", "MODEL_VERSION", "AGGREGATED_VALUE", "BIN",
                F.col("Q"))
    )

    # --- Full outer join on BIN so bins in only one distribution contribute ---
    joined = inference.join(
        baseline,
        on=["AGGREGATED_VALUE", "BIN"],
        how="full",
    )

    # Apply epsilon smoothing to avoid log(0)
    p = F.greatest(F.coalesce(F.col("P"), F.lit(0.0)), EPSILON)
    q = F.greatest(F.coalesce(F.col("Q"), F.lit(0.0)), EPSILON)
    m = (p + q) / F.lit(2.0)

    # KL(P||M) and KL(Q||M) components
    kl_p_m = p * F.ln(p / m)
    kl_q_m = q * F.ln(q / m)

    # JSD = sqrt(0.5 * KL(P||M) + 0.5 * KL(Q||M))
    jsd_df = (
        joined
        .with_column("KL_P_M", kl_p_m)
        .with_column("KL_Q_M", kl_q_m)
        .group_by("WEEK", "MODEL_VERSION", "AGGREGATED_VALUE")
        .agg(
            F.sum("KL_P_M").alias("SUM_KL_P_M"),
            F.sum("KL_Q_M").alias("SUM_KL_Q_M"),
        )
        .with_column(
            "JS_DISTANCE",
            F.sqrt(F.lit(0.5) * F.col("SUM_KL_P_M") + F.lit(0.5) * F.col("SUM_KL_Q_M")),
        )
        .with_column(
            "DATA_DATE",
            F.dateadd("week", F.col("WEEK") % F.lit(100) - F.lit(1),
                      F.date_from_parts(F.floor(F.col("WEEK") / F.lit(100)), F.lit(1), F.lit(1))),
        )
        .select(
            F.sha2(
                F.concat(
                    F.lit(MODEL_NAME), F.lit("||"),
                    F.col("MODEL_VERSION"), F.lit("||"),
                    F.object_construct(
                        F.lit("prediction_name"), F.lit(PREDICTION_COL),
                        F.lit("week"), F.col("WEEK"),
                        F.lit("data_date"), F.col("DATA_DATE"),
                    ).cast("STRING"), F.lit("||"),
                    F.lit(agg_col_lower), F.lit("||"),
                    F.col("AGGREGATED_VALUE").cast("STRING"),
                ),
                256,
            ).alias("RECORD_ID"),
            F.lit(MODEL_NAME).alias("MODEL_NAME"),
            F.col("MODEL_VERSION"),
            F.object_construct(
                F.lit("prediction_name"), F.lit(PREDICTION_COL),
                F.lit("week"), F.col("WEEK"),
                F.lit("data_date"), F.col("DATA_DATE"),
            ).alias("ENTITY_MAP"),
            F.lit(agg_col_lower).alias("AGGREGATED_COL"),
            F.col("AGGREGATED_VALUE").cast("STRING").alias("AGGREGATED_VALUE"),
            F.lit("jensen-shannon").alias("METRIC_COL"),
            F.col("JS_DISTANCE").cast("DOUBLE").alias("METRIC_VALUE"),
            F.lit(0.2).cast("DOUBLE").alias("WARNING_THRESHOLD"),
            F.lit(0.45).cast("DOUBLE").alias("CRITICAL_THRESHOLD"),
            F.when(F.col("JS_DISTANCE") > 0.45, F.lit(2))
             .when(F.col("JS_DISTANCE") > 0.2, F.lit(1))
             .otherwise(F.lit(0)).alias("ALERT_LEVEL"),
            F.lit("MXBEB").alias("BKCC"),
            F.to_char(F.col("DATA_DATE"), F.lit("YYYYMM")).alias("CALMONTH"),
            F.current_timestamp().alias("LDTS"),
        )
    )

    jsd_df.write.mode("append").save_as_table(PRED_DRIFT)

    n_rows = session.table(PRED_DRIFT).count()
    print(f"Prediction JSD for {agg_col}: PRED_DRIFT now has {n_rows} rows.")


## 4. Performance drift

In [ ]:
def compute_performance_metrics(paired_df, agg_col):
    """Compute WAPE, RMSE, MAE, and F1 from a paired predictions+actuals DataFrame.

    Parameters
    ----------
    paired_df : snowpark.DataFrame
        Must contain columns: WEEK, MODEL_VERSION, TARGET_COL, PREDICTION_COL,
        and the aggregation column *agg_col*.
    agg_col : str
        Column to group by (e.g. "STATS_NTILE_GROUP").

    Returns
    -------
    snowpark.DataFrame
        Long-format with columns: WEEK, MODEL_VERSION, AGGREGATED_COL,
        AGGREGATED_VALUE, METRIC_COL, METRIC_VALUE.
    """
    agg_col_lower = agg_col.lower()
    actual = F.col(TARGET_COL)
    predicted = F.col(PREDICTION_COL)
    abs_error = F.abs(actual - predicted)
    sq_error = F.pow(actual - predicted, F.lit(2))
    actual_positive = F.iff(actual > F.lit(0), F.lit(1), F.lit(0))
    predicted_positive = F.iff(predicted > F.lit(0), F.lit(1), F.lit(0))

    def _metrics_for_group(df, group_cols, agg_value_expr):
        """Compute the 4 metrics for a given grouping."""
        base = df.group_by(*group_cols).agg(
            # WAPE components
            F.sum(abs_error).alias("SUM_ABS_ERROR"),
            F.sum(F.abs(actual)).alias("SUM_ABS_ACTUAL"),
            # RMSE
            F.avg(sq_error).alias("AVG_SQ_ERROR"),
            # MAE
            F.avg(abs_error).alias("MAE_VAL"),
            # F1 components
            F.sum(actual_positive * predicted_positive).alias("TP"),
            F.sum((F.lit(1) - actual_positive) * predicted_positive).alias("FP"),
            F.sum(actual_positive * (F.lit(1) - predicted_positive)).alias("FN"),
        )

        wape = base.select(
            F.col("WEEK"), F.col("MODEL_VERSION"),
            F.lit(agg_col_lower).alias("AGGREGATED_COL"),
            agg_value_expr.alias("AGGREGATED_VALUE"),
            F.lit("wape").alias("METRIC_COL"),
            (F.col("SUM_ABS_ERROR") / F.greatest(F.col("SUM_ABS_ACTUAL"), F.lit(1e-10)))
                .cast("DOUBLE").alias("METRIC_VALUE"),
        )

        rmse = base.select(
            F.col("WEEK"), F.col("MODEL_VERSION"),
            F.lit(agg_col_lower).alias("AGGREGATED_COL"),
            agg_value_expr.alias("AGGREGATED_VALUE"),
            F.lit("rmse").alias("METRIC_COL"),
            F.sqrt(F.col("AVG_SQ_ERROR")).cast("DOUBLE").alias("METRIC_VALUE"),
        )

        mae = base.select(
            F.col("WEEK"), F.col("MODEL_VERSION"),
            F.lit(agg_col_lower).alias("AGGREGATED_COL"),
            agg_value_expr.alias("AGGREGATED_VALUE"),
            F.lit("mae").alias("METRIC_COL"),
            F.col("MAE_VAL").cast("DOUBLE").alias("METRIC_VALUE"),
        )

        precision = F.col("TP") / F.greatest(F.col("TP") + F.col("FP"), F.lit(1e-10))
        recall = F.col("TP") / F.greatest(F.col("TP") + F.col("FN"), F.lit(1e-10))

        f1 = base.select(
            F.col("WEEK"), F.col("MODEL_VERSION"),
            F.lit(agg_col_lower).alias("AGGREGATED_COL"),
            agg_value_expr.alias("AGGREGATED_VALUE"),
            F.lit("f1_binary").alias("METRIC_COL"),
            (F.lit(2) * precision * recall / F.greatest(precision + recall, F.lit(1e-10)))
                .cast("DOUBLE").alias("METRIC_VALUE"),
        )

        return wape.union_all_by_name(rmse).union_all_by_name(mae).union_all_by_name(f1)

    # Per aggregation value
    per_segment = _metrics_for_group(
        paired_df,
        ["WEEK", "MODEL_VERSION", agg_col],
        F.col(agg_col).cast("STRING"),
    )

    # Full model (across all aggregation values)
    full_model = _metrics_for_group(
        paired_df,
        ["WEEK", "MODEL_VERSION"],
        F.lit("full_model"),
    )

    return per_segment.union_all_by_name(full_model)

In [ ]:
# Get weeks where actuals are available (used to filter combos for performance metrics)
actuals_weeks = (
    session.table(INFERENCE_ACTUALS)
    .select(F.col(TIME_COL).alias("WEEK"))
    .distinct()
    .collect()
)
actuals_week_set = {row["WEEK"] for row in actuals_weeks}
print(f"INFERENCE_ACTUALS has {len(actuals_week_set)} distinct weeks.")

# Calculate baseline performance metrics — loop over each aggregation column

perf_new_combos_list = {}

for agg_col in AGG_COLS:
    new_baseline_versions, all_combos = get_new_combos(
        PERF_BASELINE, PERF_DRIFT, agg_col=agg_col, metric_col=PERF_METRIC_NAMES[0])

    # Filter to only weeks where actuals exist
    perf_new_combos_list[agg_col] = [
        c for c in all_combos if c["WEEK"] in actuals_week_set
    ]
    print(f"  After actuals filter: {len(perf_new_combos_list[agg_col])} combos for {agg_col}")

    if not new_baseline_versions:
        print(f"  No new baseline versions for {agg_col}, skipping.")
        continue

    for version in new_baseline_versions:
        # Join baseline predictions with baseline actuals
        preds = (
            session.table(BASELINE_PREDICTIONS)
            .filter(F.col("MODEL_VERSION") == version)
        )
        actuals = session.table(BASELINE_DATA).drop(AGG_COLS)

        paired = preds.join(actuals, on=PERF_JOIN_KEYS, how="inner")

        # Compute metrics
        metrics_df = compute_performance_metrics(paired, agg_col)

        # Shape into PERF_BASELINE schema
        data_date_expr = F.dateadd(
            "week", F.col("WEEK") % F.lit(100) - F.lit(1),
            F.date_from_parts(F.floor(F.col("WEEK") / F.lit(100)), F.lit(1), F.lit(1)),
        )

        baseline_rows = (
            metrics_df
            .with_column("DATA_DATE", data_date_expr)
            .select(
                F.sha2(
                    F.concat(
                        F.lit(MODEL_NAME), F.lit("||"),
                        F.col("MODEL_VERSION"), F.lit("||"),
                        F.object_construct(
                            F.lit("week"), F.col("WEEK"),
                            F.lit("data_date"), F.col("DATA_DATE"),
                        ).cast("STRING"), F.lit("||"),
                        F.col("AGGREGATED_COL"), F.lit("||"),
                        F.col("AGGREGATED_VALUE"), F.lit("||"),
                        F.col("METRIC_COL"),
                    ),
                    256,
                ).alias("RECORD_ID"),
                F.lit(MODEL_NAME).alias("MODEL_NAME"),
                F.col("MODEL_VERSION"),
                F.object_construct(
                    F.lit("week"), F.col("WEEK"),
                    F.lit("data_date"), F.col("DATA_DATE"),
                ).alias("ENTITY_MAP"),
                F.col("AGGREGATED_COL"),
                F.col("AGGREGATED_VALUE"),
                F.col("METRIC_COL"),
                F.col("METRIC_VALUE"),
                F.lit(None).cast("DOUBLE").alias("METRIC_DRIFT"),
                F.lit(0.0).cast("DOUBLE").alias("WARNING_THRESHOLD"),
                F.lit(0.0).cast("DOUBLE").alias("CRITICAL_THRESHOLD"),
                F.lit(0).alias("ALERT_LEVEL"),
                F.lit("MXBEB").alias("BKCC"),
                F.to_char(F.col("DATA_DATE"), F.lit("YYYYMM")).alias("CALMONTH"),
                F.current_timestamp().alias("LDTS"),
            )
        )

        baseline_rows.write.mode("append").save_as_table(PERF_BASELINE)

    n_rows = session.table(PERF_BASELINE).count()
    print(f"Baseline perf for {agg_col}: PERF_BASELINE now has {n_rows} rows.")


In [ ]:
# Calculate inference performance metrics and drift — loop over each aggregation column

# Thresholds for proportional drift (METRIC_DRIFT column)
# For error metrics (higher = worse): warn at +20%, critical at +50%
# For F1 (lower = worse): warn at -15%, critical at -30%
PERF_THRESHOLDS = {
    "wape":      {"warn": 0.20, "crit": 0.50, "direction": "higher_worse"},
    "rmse":      {"warn": 0.20, "crit": 0.50, "direction": "higher_worse"},
    "mae":       {"warn": 0.20, "crit": 0.50, "direction": "higher_worse"},
    "f1_binary": {"warn": -0.15, "crit": -0.30, "direction": "lower_worse"},
}

for agg_col in AGG_COLS:
    if not perf_new_combos_list[agg_col]:
        print(f"  No new inference combos for {agg_col}, skipping.")
        continue

    # Read baseline averages for this agg_col
    baseline_avg = (
        session.table(PERF_BASELINE)
        .filter(F.col("AGGREGATED_COL") == agg_col.lower())
        .group_by("MODEL_VERSION", "AGGREGATED_COL", "AGGREGATED_VALUE", "METRIC_COL")
        .agg(F.avg("METRIC_VALUE").alias("BASELINE_METRIC"))
    )

    frames = []
    for combo in perf_new_combos_list[agg_col]:
        week, version = combo["WEEK"], combo["MODEL_VERSION"]

        preds = (
            session.table(INFERENCE_PREDICTIONS)
            .filter(F.col("MODEL_VERSION") == version)
            .filter(F.col(TIME_COL) == week)
        )
        actuals = (
            session.table(INFERENCE_ACTUALS)
            .filter(F.col(TIME_COL) == week)
            .drop(AGG_COLS)
        )

        paired = preds.join(actuals, on=PERF_JOIN_KEYS, how="inner")
        metrics_df = compute_performance_metrics(paired, agg_col)
        frames.append(metrics_df)

    all_metrics = frames[0]
    for f in frames[1:]:
        all_metrics = all_metrics.union_all_by_name(f)

    # Join with baseline averages to compute drift
    with_drift = all_metrics.join(
        baseline_avg,
        on=["MODEL_VERSION", "AGGREGATED_COL", "AGGREGATED_VALUE", "METRIC_COL"],
        how="left",
    ).with_column(
        "METRIC_DRIFT",
        (F.col("METRIC_VALUE") - F.col("BASELINE_METRIC"))
        / F.greatest(F.abs(F.col("BASELINE_METRIC")), F.lit(1e-10)),
    )

    data_date_expr = F.dateadd(
        "week", F.col("WEEK") % F.lit(100) - F.lit(1),
        F.date_from_parts(F.floor(F.col("WEEK") / F.lit(100)), F.lit(1), F.lit(1)),
    )

    # Build per-metric threshold and alert columns using CASE expressions
    warn_expr = (
        F.when(F.col("METRIC_COL") == "f1_binary", F.lit(PERF_THRESHOLDS["f1_binary"]["warn"]))
         .otherwise(F.lit(PERF_THRESHOLDS["wape"]["warn"]))
         .cast("DOUBLE")
    )
    crit_expr = (
        F.when(F.col("METRIC_COL") == "f1_binary", F.lit(PERF_THRESHOLDS["f1_binary"]["crit"]))
         .otherwise(F.lit(PERF_THRESHOLDS["wape"]["crit"]))
         .cast("DOUBLE")
    )

    # Alert: for error metrics drift > threshold is bad; for F1 drift < threshold is bad
    alert_expr = (
        F.when(
            F.col("METRIC_COL") == "f1_binary",
            F.when(F.col("METRIC_DRIFT") < F.lit(PERF_THRESHOLDS["f1_binary"]["crit"]), F.lit(2))
             .when(F.col("METRIC_DRIFT") < F.lit(PERF_THRESHOLDS["f1_binary"]["warn"]), F.lit(1))
             .otherwise(F.lit(0))
        ).otherwise(
            F.when(F.col("METRIC_DRIFT") > F.lit(PERF_THRESHOLDS["wape"]["crit"]), F.lit(2))
             .when(F.col("METRIC_DRIFT") > F.lit(PERF_THRESHOLDS["wape"]["warn"]), F.lit(1))
             .otherwise(F.lit(0))
        )
    )

    drift_rows = (
        with_drift
        .with_column("DATA_DATE", data_date_expr)
        .select(
            F.sha2(
                F.concat(
                    F.lit(MODEL_NAME), F.lit("||"),
                    F.col("MODEL_VERSION"), F.lit("||"),
                    F.object_construct(
                        F.lit("week"), F.col("WEEK"),
                        F.lit("data_date"), F.col("DATA_DATE"),
                    ).cast("STRING"), F.lit("||"),
                    F.col("AGGREGATED_COL"), F.lit("||"),
                    F.col("AGGREGATED_VALUE"), F.lit("||"),
                    F.col("METRIC_COL"),
                ),
                256,
            ).alias("RECORD_ID"),
            F.lit(MODEL_NAME).alias("MODEL_NAME"),
            F.col("MODEL_VERSION"),
            F.object_construct(
                F.lit("week"), F.col("WEEK"),
                F.lit("data_date"), F.col("DATA_DATE"),
            ).alias("ENTITY_MAP"),
            F.col("AGGREGATED_COL"),
            F.col("AGGREGATED_VALUE"),
            F.col("METRIC_COL"),
            F.col("METRIC_VALUE").cast("DOUBLE"),
            F.col("METRIC_DRIFT").cast("DOUBLE"),
            warn_expr.alias("WARNING_THRESHOLD"),
            crit_expr.alias("CRITICAL_THRESHOLD"),
            alert_expr.alias("ALERT_LEVEL"),
            F.lit("MXBEB").alias("BKCC"),
            F.to_char(F.col("DATA_DATE"), F.lit("YYYYMM")).alias("CALMONTH"),
            F.current_timestamp().alias("LDTS"),
        )
    )

    drift_rows.write.mode("append").save_as_table(PERF_DRIFT)

    n_rows = session.table(PERF_DRIFT).count()
    print(f"Perf drift for {agg_col}: PERF_DRIFT now has {n_rows} rows.")